# UR5 SmolVLA training on Google Colab Pro

This notebook performs a **pipeline smoke test**, not autonomous deployment. It never contacts ROS or the robot, never trains directly from mounted Drive, and never uploads to Hugging Face. Ten episodes are not evidence of a reliable policy; collect roughly 50 or more high-quality demonstrations after the pipeline is proven.

Before starting, change the shared Drive folder from **Anyone with the link: Editor** to **Restricted**, explicitly add the Colab account as Editor, and add a shortcut to the folder in My Drive.

## A — Runtime verification

Capture Python, OS, GPU, CUDA, PyTorch, VRAM, and RAM. Stop if Colab did not assign a CUDA GPU.

In [1]:
import json, os, platform, subprocess, sys, time
from pathlib import Path

def command_output(command):
    return subprocess.run(command, text=True, capture_output=True, check=False).stdout.strip()

runtime_report = {
    "python": sys.version,
    "os": platform.platform(),
    "nvidia_smi": command_output(["nvidia-smi"]),
}
print(json.dumps(runtime_report, indent=2))
try:
    import torch
    runtime_report.update({
        "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0,
        "gpu_free_total_bytes": torch.cuda.mem_get_info() if torch.cuda.is_available() else (0, 0),
    })
except Exception as exc:
    runtime_report["torch_error"] = repr(exc)
print(json.dumps(runtime_report, indent=2))
assert runtime_report.get("cuda_available"), "No CUDA GPU assigned: Runtime > Change runtime type > GPU"

{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]",
  "os": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "nvidia_smi": "Sat Aug 22 05:58:47 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |\n|                                         |                        |               MIG M. |\n|=========================================+========================+======================|\n|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |\n| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default 

## B — Mount and configure Google Drive

Edit only `DRIVE_ROOT` if your shortcut has a different name. The checks deliberately stop before any missing archive is used.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/pHYSICAL UR5")
DATASET_ARCHIVE = DRIVE_ROOT / "datasets/lerobot/ur5_orange_cube_v1.tar"
DATASET_CHECKSUM = Path(f"{DATASET_ARCHIVE}.sha256")
DATASET_PARTS = sorted(DATASET_ARCHIVE.parent.glob(f"{DATASET_ARCHIVE.name}.part-*"))
DATASET_PARTS_CHECKSUM = Path(f"{DATASET_ARCHIVE}.parts.sha256")
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints/ur5_smolvla_test"
REPORT_ROOT = DRIVE_ROOT / "reports/training"
DEPLOYMENT_ROOT = DRIVE_ROOT / "deployment/downloaded_checkpoints"

assert DRIVE_ROOT.is_dir(), f"Drive shortcut/path not found: {DRIVE_ROOT}"
assert DATASET_ARCHIVE.is_file() or DATASET_PARTS, f"Dataset archive or split parts missing beside: {DATASET_ARCHIVE}"
assert DATASET_CHECKSUM.is_file(), f"Checksum missing: {DATASET_CHECKSUM}"
if not DATASET_ARCHIVE.is_file(): assert DATASET_PARTS_CHECKSUM.is_file(), "Split-part checksum file missing"
for directory in (CHECKPOINT_ROOT, REPORT_ROOT, DEPLOYMENT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
    probe = directory / ".write_test"; probe.write_text("ok"); probe.unlink()
(REPORT_ROOT / "runtime_report.json").write_text(json.dumps(runtime_report, indent=2))
print("Drive configuration verified")

AssertionError: Drive shortcut/path not found: /content/drive/MyDrive/pHYSICAL UR5

## C — Install and inspect the pinned training environment

LeRobot is pinned to 0.4.3. PyAV is the explicit reliable video backend. TorchCodec is tested but is not required if its wheel and the Colab FFmpeg ABI differ. A runtime restart may be required only if pip replaces an already-imported PyTorch package; if so, restart and rerun from Section A.

In [ ]:
%pip install -q --upgrade pip
%pip install -q "lerobot[smolvla]==0.4.3" "av>=12" psutil matplotlib pytest
!ffmpeg -version | head -2

In [ ]:
import importlib.metadata as md, subprocess, sys
for package in ("lerobot", "torch", "torchvision", "av", "transformers", "accelerate"):
    try: print(package, md.version(package))
    except md.PackageNotFoundError: print(package, "NOT INSTALLED")
print(subprocess.run(["lerobot-train", "--help"], text=True, capture_output=True).stdout[:12000])
try:
    import torchcodec
    print("TorchCodec import PASS", md.version("torchcodec"))
except Exception as exc:
    print("TorchCodec unavailable/incompatible; using supported PyAV fallback:", repr(exc))
import av
print("PyAV PASS", av.__version__)

## D — Copy, verify, and safely extract the dataset

Training occurs under `/content`, never on mounted Drive. SHA-256 is checked before extraction; absolute paths, traversal, and links are rejected.

In [ ]:
import hashlib, shutil, tarfile
from pathlib import Path, PurePosixPath

LOCAL_ARCHIVE = Path("/content/ur5_orange_cube_v1.tar")
LOCAL_DATASET = Path("/content/ur5_orange_cube_v1")
if DATASET_ARCHIVE.is_file():
    shutil.copy2(DATASET_ARCHIVE, LOCAL_ARCHIVE)
else:
    expected_part_hashes={line.split()[1]:line.split()[0] for line in DATASET_PARTS_CHECKSUM.read_text().splitlines() if line.strip()}
    with LOCAL_ARCHIVE.open("wb") as assembled:
        for part in DATASET_PARTS:
            part_digest=hashlib.sha256(part.read_bytes()).hexdigest()
            assert part_digest==expected_part_hashes[part.name], f"part SHA-256 mismatch: {part.name}"
            with part.open("rb") as source: shutil.copyfileobj(source,assembled,length=8*1024*1024)
    print("Reassembled split Drive upload into",LOCAL_ARCHIVE)
expected_sha = DATASET_CHECKSUM.read_text().split()[0].lower()
digest = hashlib.sha256()
with LOCAL_ARCHIVE.open("rb") as stream:
    while chunk := stream.read(8 * 1024 * 1024): digest.update(chunk)
actual_sha = digest.hexdigest()
assert actual_sha == expected_sha, f"SHA-256 mismatch: {actual_sha} != {expected_sha}"
if LOCAL_DATASET.exists(): shutil.rmtree(LOCAL_DATASET)
with tarfile.open(LOCAL_ARCHIVE, "r:") as archive:
    members = archive.getmembers()
    for member in members:
        path = PurePosixPath(member.name)
        assert not path.is_absolute() and ".." not in path.parts, f"unsafe path: {member.name}"
        assert not member.issym() and not member.islnk(), f"archive link rejected: {member.name}"
    archive.extractall("/content", members=members, filter="data")
assert LOCAL_DATASET.is_dir()
print("Archive verified and extracted:", LOCAL_DATASET)

### Load and validate through LeRobot

Validate metadata and every state/action row, then decode representative synchronized views from every episode.

In [ ]:
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

REPO_ID = "local/ur5_orange_cube_v1"
TASK = "Pick up the orange cube and place it in the tray"
dataset = LeRobotDataset(REPO_ID, root=LOCAL_DATASET, video_backend="pyav")
features = dataset.meta.features
assert dataset.meta.total_episodes == 10 and dataset.meta.total_frames == 20583
assert features["observation.images.base"]["dtype"] == "video"
assert features["observation.images.wrist"]["dtype"] == "video"
assert tuple(features["observation.state"]["shape"]) == (7,)
assert tuple(features["action"]["shape"]) == (7,)
assert "source_episode_time_s" in features and TASK in dataset.meta.tasks.index
for index in range(len(dataset)):
    row = dataset.hf_dataset[index]
    state, action = np.asarray(row["observation.state"]), np.asarray(row["action"])
    assert np.isfinite(state).all() and np.isfinite(action).all()
    assert min(abs(float(state[6])), abs(float(state[6])-1)) < 1e-5
    assert min(abs(float(action[6])), abs(float(action[6])-1)) < 1e-5
print("Dataset metadata/vector validation PASS", dataset.meta.total_episodes, dataset.meta.total_frames)

### Display one synchronized sample from every episode

In [ ]:
import matplotlib.pyplot as plt
episode_rows = dataset.meta.episodes
fig, axes = plt.subplots(10, 2, figsize=(12, 35))
representative_samples = []
for episode in range(10):
    start, stop = int(episode_rows[episode]["dataset_from_index"]), int(episode_rows[episode]["dataset_to_index"])
    index = (start + stop - 1) // 2
    sample = dataset[index]; representative_samples.append(sample)
    for column, key in enumerate(("observation.images.base", "observation.images.wrist")):
        image = sample[key].permute(1, 2, 0).cpu().numpy()
        axes[episode, column].imshow(image)
        axes[episode, column].set_title(f"ep={episode} frame={int(sample['frame_index'])} {key}")
        axes[episode, column].axis("off")
    print({"episode": episode, "frame": int(sample["frame_index"]), "timestamp": float(sample["timestamp"]),
           "task": TASK, "state": sample["observation.state"].tolist(), "action": sample["action"].tolist(),
           "gripper_state": float(sample["observation.state"][6]), "gripper_command": float(sample["action"][6])})
plt.tight_layout(); plt.show()

## E — Resolve and save the conservative training configuration

Start at batch size 1, fixed seed, no WandB, no Hub push, and the pretrained `lerobot/smolvla_base`. The CLI help above is the authority for this pinned installation.

In [ ]:
import json, os, random
import numpy as np, torch

SEED = 42; BATCH_SIZE = 1; GRAD_ACCUMULATION = 1
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
LOCAL_OUTPUT = Path("/content/outputs/ur5_smolvla_test")
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
BASE_ARGS = [
    "lerobot-train", "--policy.type=smolvla", "--policy.pretrained_path=lerobot/smolvla_base",
    "--policy.device=cuda", "--policy.use_amp=true", "--policy.push_to_hub=false",
    f"--dataset.repo_id={REPO_ID}", f"--dataset.root={LOCAL_DATASET}", "--dataset.video_backend=pyav",
    f"--batch_size={BATCH_SIZE}", f"--seed={SEED}", "--num_workers=2", "--wandb.enable=false",
]
resolved_training = {"args": BASE_ARGS, "batch_size": BATCH_SIZE, "gradient_accumulation": GRAD_ACCUMULATION,
                     "seed": SEED, "rgb_only": True, "task": TASK}
(LOCAL_OUTPUT / "resolved_training_config.json").write_text(json.dumps(resolved_training, indent=2))
print(json.dumps(resolved_training, indent=2))

## F — Pre-training tests

Each stage stops on failure and records time, CPU RAM, peak VRAM, loss/NaN status when available, and video errors. The one-step CLI run is the authoritative end-to-end forward/backward/update test for LeRobot 0.4.3.

In [ ]:
import psutil, time, torch
from torch.utils.data import DataLoader
preflight = {}
def record(name, started, **values):
    values.update({"pass": True, "step_time_s": time.time()-started,
                   "cpu_ram_bytes": psutil.Process().memory_info().rss,
                   "peak_gpu_vram_bytes": torch.cuda.max_memory_allocated()})
    preflight[name] = values; print(name, values)

started=time.time(); _=dataset[0]; record("dataset_loader", started)
started=time.time(); batch=next(iter(DataLoader(dataset,batch_size=1,num_workers=0))); record("one_batch",started,keys=list(batch))

### Model loading and a no-gradient forward pass

In [ ]:
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.factory import make_policy, make_pre_post_processors

torch.cuda.reset_peak_memory_stats(); started=time.time()
cfg=PreTrainedConfig.from_pretrained("lerobot/smolvla_base"); cfg.device="cuda"; cfg.use_amp=True; cfg.push_to_hub=False
policy=make_policy(cfg,ds_meta=dataset.meta).eval()
preprocessor,postprocessor=make_pre_post_processors(cfg,dataset_stats=dataset.meta.stats)
record("model_loading",started)
started=time.time()
model_batch={key:(value.cuda() if isinstance(value,torch.Tensor) else value) for key,value in batch.items()}
model_batch["task"]=[TASK]
processed=preprocessor(model_batch)
with torch.inference_mode(): loss,details=policy.forward(processed)
assert torch.isfinite(loss); record("forward_pass",started,loss=float(loss),nan_inf=False)
del processed, model_batch; torch.cuda.empty_cache()

### One training step

This writes to Colab local disk only.

In [ ]:
import re, subprocess, time
def last_reported_loss(text):
    matches=re.findall(r"(?:loss[=: ]+)([-+0-9.eE]+)",text,re.IGNORECASE)
    return float(matches[-1]) if matches else None
ONE_STEP_OUTPUT=Path("/content/outputs/preflight_one_step")
started=time.time(); torch.cuda.reset_peak_memory_stats()
result=subprocess.run(BASE_ARGS+["--steps=1",f"--output_dir={ONE_STEP_OUTPUT}","--job_name=preflight_one_step","--save_freq=1"],text=True,capture_output=True)
print(result.stdout[-12000:]); print(result.stderr[-4000:]); assert result.returncode==0
record("one_training_step",started,returncode=result.returncode,loss=last_reported_loss(result.stdout),nan_inf="nan" in result.stdout.lower() or "inf" in result.stdout.lower())

### Ten-step smoke training

This is the authorized notebook smoke test. It remains local until the following checkpoint-copy cell.

In [ ]:
SMOKE_OUTPUT=Path("/content/outputs/ur5_smolvla_smoke_10")
started=time.time(); torch.cuda.reset_peak_memory_stats()
result=subprocess.run(BASE_ARGS+["--steps=10",f"--output_dir={SMOKE_OUTPUT}","--job_name=ur5_smolvla_smoke_10","--save_freq=10"],text=True,capture_output=True)
(SMOKE_OUTPUT/"captured_stdout.log").write_text(result.stdout); (SMOKE_OUTPUT/"captured_stderr.log").write_text(result.stderr)
print(result.stdout[-12000:]); print(result.stderr[-4000:]); assert result.returncode==0
record("ten_step_smoke",started,returncode=result.returncode,loss=last_reported_loss(result.stdout),nan_inf="nan" in result.stdout.lower() or "inf" in result.stdout.lower())
(LOCAL_OUTPUT/"preflight_report.json").write_text(json.dumps(preflight,indent=2))

## G — Reload, infer, compare, and save the complete smoke checkpoint

Copy the **entire** checkpoint plus reports—not only `model.safetensors`—to Drive.

In [ ]:
import glob, shutil
models=sorted(SMOKE_OUTPUT.glob("checkpoints/*/pretrained_model"),key=lambda p:p.stat().st_mtime)
assert models, "No complete pretrained_model directory found"
SMOKE_MODEL=models[-1]
required_any=("model.safetensors","config.json","preprocessor_config.json","postprocessor_config.json")
print("checkpoint files",[p.name for p in SMOKE_MODEL.iterdir()]); assert (SMOKE_MODEL/"model.safetensors").is_file()
drive_smoke=CHECKPOINT_ROOT/"smoke_10_steps"
if drive_smoke.exists(): shutil.rmtree(drive_smoke)
shutil.copytree(SMOKE_MODEL,drive_smoke)
shutil.copy2(LOCAL_OUTPUT/"preflight_report.json",REPORT_ROOT/"preflight_report.json")
shutil.copy2(LOCAL_OUTPUT/"resolved_training_config.json",REPORT_ROOT/"resolved_training_config.json")
print("Complete smoke checkpoint copied to",drive_smoke)

### Checkpoint reload and recorded-observation inference

In [ ]:
from lerobot.policies.factory import get_policy_class
reload_cfg=PreTrainedConfig.from_pretrained(SMOKE_MODEL,local_files_only=True); reload_cfg.device="cuda"
reloaded=get_policy_class(reload_cfg.type).from_pretrained(SMOKE_MODEL,config=reload_cfg,local_files_only=True).cuda().eval()
reload_pre,reload_post=make_pre_post_processors(reload_cfg,pretrained_path=SMOKE_MODEL,dataset_stats=dataset.meta.stats)
demo=[];pred=[]
with torch.inference_mode():
    for index in np.linspace(0,len(dataset)-1,50,dtype=int):
        sample=dataset[int(index)]; model_input={k:v.unsqueeze(0).cuda() for k,v in sample.items() if isinstance(v,torch.Tensor)}; model_input["task"]=[TASK]
        chunk=reloaded.predict_action_chunk(reload_pre(model_input)); values=reload_post(chunk).detach().cpu().numpy()
        if values.ndim==2: values=values[None]
        assert values.ndim==3 and values.shape[-1]==7 and np.isfinite(values).all()
        demo.append(sample["action"].numpy()); pred.append(values[0,0])
demo=np.asarray(demo); pred=np.asarray(pred); jumps=np.max(np.abs(np.diff(pred[:,:6],axis=0)),axis=0)
inference_report={"shape":list(pred.shape),"predicted_min":pred.min(0).tolist(),"predicted_max":pred.max(0).tolist(),"max_joint_jump":jumps.tolist(),"gripper_predictions":pred[:,6].tolist()}
(REPORT_ROOT/"smoke_inference_report.json").write_text(json.dumps(inference_report,indent=2)); print(json.dumps(inference_report,indent=2))
fig,axes=plt.subplots(7,1,figsize=(12,16),sharex=True)
for joint,axis in enumerate(axes): axis.plot(demo[:,joint],label="demonstrated");axis.plot(pred[:,joint],label="predicted");axis.set_ylabel(str(joint));axis.legend()
plt.tight_layout(); fig.savefig(REPORT_ROOT/"smoke_demo_vs_prediction.png",dpi=150); plt.show()

## H — Optional full 5,000-step experiment (manual opt-in)

This cell refuses to run until `RUN_FULL_TRAINING=True`. It estimates runtime from the ten-step smoke test, writes checkpoints locally first, and then copies complete checkpoint directories to Drive. LeRobot 0.4.3 does not expose gradient accumulation in the inspected CLI, so keep it at 1 unless a later inspected version supports it.

In [ ]:
RUN_FULL_TRAINING=False
if not RUN_FULL_TRAINING: raise RuntimeError("Full training disabled. Review smoke results, then explicitly set RUN_FULL_TRAINING=True.")
smoke_seconds=preflight["ten_step_smoke"]["step_time_s"];print("Estimated 5000-step hours",smoke_seconds/10*5000/3600)
FULL_OUTPUT=Path("/content/outputs/ur5_smolvla_5000")
full_args=BASE_ARGS+["--steps=5000",f"--output_dir={FULL_OUTPUT}","--job_name=ur5_smolvla_5000","--save_freq=250","--log_freq=10"]
print("Resolved full command",full_args)
def sync_complete_checkpoints():
    for model in FULL_OUTPUT.glob("checkpoints/*/pretrained_model"):
        if not (model/"model.safetensors").is_file() or not (model/"config.json").is_file(): continue
        target=CHECKPOINT_ROOT/"full_5000"/model.parent.name/"pretrained_model"; target.parent.mkdir(parents=True,exist_ok=True)
        staging=target.with_name(".pretrained_model.copying")
        if target.exists(): continue
        if staging.exists(): shutil.rmtree(staging)
        shutil.copytree(model,staging); staging.rename(target); print("Synced complete checkpoint",target)
process=subprocess.Popen(full_args,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
with (LOCAL_OUTPUT/"full_training.log").open("w") as log:
    while process.poll() is None:
        line=process.stdout.readline()
        if line: print(line,end="");log.write(line);log.flush()
        sync_complete_checkpoints()
    for line in process.stdout: print(line,end="");log.write(line)
assert process.returncode==0;sync_complete_checkpoints();shutil.copy2(LOCAL_OUTPUT/"full_training.log",REPORT_ROOT/"full_training.log")

## I — Resume from the latest complete Drive checkpoint

This cell verifies a complete `pretrained_model`, copies its whole checkpoint parent locally, and uses LeRobot's resume flag so training cannot silently restart at step zero.

In [ ]:
RESUME_TRAINING=False
if not RESUME_TRAINING: raise RuntimeError("Resume disabled. Set RESUME_TRAINING=True after selecting/reviewing Drive checkpoints.")
candidates=sorted((CHECKPOINT_ROOT/"full_5000").glob("*/pretrained_model"),key=lambda p:p.parent.name)
assert candidates; source_model=candidates[-1]; assert (source_model/"model.safetensors").is_file() and (source_model/"config.json").is_file()
resume_parent=Path("/content/outputs/resume")/source_model.parent.name
if resume_parent.exists(): shutil.rmtree(resume_parent)
shutil.copytree(source_model.parent,resume_parent)
resumed_step=int(''.join(filter(str.isdigit,source_model.parent.name)) or -1); assert resumed_step>=0
resume_args=BASE_ARGS+["--steps=5000",f"--output_dir={resume_parent.parent}","--job_name=ur5_smolvla_5000","--resume=true"]
print("Resuming from step",resumed_step,"command",resume_args)
result=subprocess.run(resume_args,text=True); assert result.returncode==0

## J — Package a complete checkpoint for the robot PC

`model.safetensors` alone is insufficient: loading also needs model configuration, preprocessing/postprocessing configuration, normalization statistics, feature information, and processor/tokenizer assets.

In [ ]:
import hashlib, tarfile
EXPORT_MODEL=SMOKE_MODEL  # change to the reviewed final full-training pretrained_model
required=["model.safetensors","config.json"]
assert all((EXPORT_MODEL/name).is_file() for name in required)
export_archive=DEPLOYMENT_ROOT/f"{EXPORT_MODEL.parent.name}_complete_checkpoint.tar"
with tarfile.open(export_archive,"w") as archive: archive.add(EXPORT_MODEL,arcname="pretrained_model")
digest=hashlib.sha256()
with export_archive.open("rb") as stream:
    while chunk:=stream.read(8*1024*1024): digest.update(chunk)
checksum=Path(f"{export_archive}.sha256");checksum.write_text(f"{digest.hexdigest()}  {export_archive.name}\n")
print("Complete checkpoint archive",export_archive,"SHA-256",digest.hexdigest())

## Stop here

Download the complete checkpoint archive and checksum to the Ubuntu PC, verify and extract it, run offline inference, and only then enter ROS shadow mode. Live mode remains locked. Never infer physical grasp success from the estimated pneumatic gripper state.